<a href="https://colab.research.google.com/github/HumzaW245/LabelShiftExperiments/blob/versionA/Trials_H2T_coding_head2toeSetupAttempt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***h2t replicating attempt***



In [1]:
import torchvision.models as models

from numpy.random import RandomState
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import Subset


from torchvision import datasets, transforms
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
def train(model, device, train_loader, optimizer, epoch, display=True):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
    if display:
      print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
          epoch, batch_idx * len(data), len(train_loader.dataset),
          100. * batch_idx / len(train_loader), loss.item()))

def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, size_average=False).item() # sum up batch loss
            pred = output.max(1, keepdim=True)[1] # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))
    return 100. * correct / len(test_loader.dataset)

In [18]:
class Net(torch.nn.Module):
    def __init__(self, datasetName, finetune_backbone):
        super(Net, self).__init__()
        self.model = models.resnet50(pretrained=True)
        self.finetune_backbone = finetune_backbone
        if(self.finetune_backbone == False):
          freezeBackbone(self.model)

        in_features = self.model.fc.in_features #The fc layer of resenet50 is Linear(in_features=2048, out_features=1000, bias=True) so storing the 2048 and replacing this to map from 2048 to numClasses for target task ====can see the fc layer like this: backbone = models.resnet50(pretrained=True) => print(backbone.fc)
        targetTaskOutFeatures = numUniqueClasses(datasetName) # num of classes in target task

        #New output head
        classifier = nn.Linear(in_features, targetTaskOutFeatures, bias=True)  # Create a new classifier
        self.model.fc = classifier  # Replace the classifier layer

        # Register the hook to each layer
        for name, module in self.model.named_modules():
            module.register_forward_hook(self.print_layer_output)

    def forward(self, x):
        return self.model(x)

    def print_layer_output(self, module, input, output):
      if module.__class__.__name__ == "Linear":
        print(f"Output shape of {module.__class__.__name__}: {output}")

In [4]:

from torchvision import datasets,transforms
import torch
import numpy as np


import torchvision.transforms as transforms
from torchvision.datasets import SVHN

# Define the transforms to apply (--------------------------------------PREPROCESSING FOR EACH DATASET WHAT IS BEST TO GO WITH IMAGENETR50)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create an instance of the SVHN dataset with the transforms
trainData = datasets.SVHN('../data', split='train', download=True, transform=transform)

testData = datasets.SVHN('../data', split='test', download=True, transform=transform)


train_loader = torch.utils.data.DataLoader(trainData,
                                           batch_size=256,
                                           shuffle=True,
                                           drop_last=True)

test_dataloader = torch.utils.data.DataLoader(testData,
                                          batch_size=256,
                                          shuffle=True,
                                          drop_last=True) #Drop last is really just to make sure if last batch is not of equal size, drop it. Nothing to do with setting it aside for testing


100%|██████████| 182040794/182040794 [00:08<00:00, 21331251.76it/s]


100%|██████████| 64275384/64275384 [00:02<00:00, 24377171.19it/s]


In [5]:
def numUniqueClasses(datasetName):

  datasetsClasses = {'SVHN': 10}
  print(f'dataset {datasetName} has {datasetsClasses[datasetName]} unique classes')
  return datasetsClasses[datasetName]

def freezeBackbone(backbone):
  for i, param in enumerate(backbone.parameters()):
    param.requires_grad = False

# model = models.resnet50(pretrained=True)
# freezeBackbone(model)

# print(model)

In [14]:
def evaluate(datasetName, finetune_backbone=False):
  use_cuda = torch.cuda.is_available()
  device = torch.device("cuda" if use_cuda else "cpu")
  print(device) # you will really need gpu's for this part




  accs = []


  # Load the pre-trained ResNet-50 model
  model = Net(datasetName, finetune_backbone)




  optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

  model.to(device)
  for epoch in range(5):
    train(model, device, train_loader, optimizer, epoch, display=True)

  accs.append(test(model, device, test_dataloader))

  accs = np.array(accs)
  print('Acc over 1 instances: %.2f +- %.2f'%(accs.mean(),accs.std()))

In [19]:
evaluate('SVHN')

cuda
dataset SVHN has 10 unique classes
Output shape of Linear: tensor([[-1.4036e-01,  4.9066e-02,  4.7253e-01,  ..., -3.2529e-02,
         -5.1764e-02,  2.6259e-01],
        [-1.7788e-01,  7.3527e-02,  6.7721e-01,  ..., -4.4599e-01,
         -3.6885e-02,  6.2377e-01],
        [-1.7317e-01, -2.7461e-01,  6.9325e-01,  ..., -3.3882e-01,
         -2.4009e-01,  4.4878e-01],
        ...,
        [ 7.1663e-02, -3.2049e-01,  7.5287e-01,  ..., -5.6745e-04,
          9.1111e-02,  5.4277e-01],
        [ 8.8779e-02, -3.8563e-01,  9.0141e-01,  ..., -1.0424e-01,
         -1.1272e-01,  4.6227e-01],
        [-8.4936e-02,  1.7320e-01,  9.6835e-01,  ..., -2.3002e-01,
          4.0689e-02,  5.4414e-01]], device='cuda:0', grad_fn=<AddmmBackward0>)
Output shape of Linear: tensor([[-0.4113,  0.3075,  0.7994,  ..., -0.4626,  0.0572,  0.2115],
        [-0.4223,  0.1862,  0.7277,  ..., -0.3782, -0.1504,  0.0225],
        [-0.0961,  0.5572,  0.5836,  ..., -0.4196, -0.0771,  0.1236],
        ...,
        [-0.58

KeyboardInterrupt: ignored

# TODO:

[x]- Transfer to vscode as is IN CLUSTER +++++ GIT REPO

[x]- RUN TO REPRODUCE ABOVE RESULTS FIRST

[x]- Once working, check FT also working

[x]- Then, rework functions to use a config file so cleaner and easier to change stuff...

- setup wandb

- Reproduce results to best ability for 4 datasets already in wandb.

- Once cleanly setup, implement basic head2toe functionality in separate file
  ****************HARD CODE AS MUCH AS NEEDED TO GET GENERAL H2T functionality tested...can define functions in detail later...FIRST JUST SETUP RANDOM FRACTION OF FEATURES TO GRAB or indices of neurons or w.e....FOCUS ON JUST GETTING SOME TRAINING DONE WITH INTERMEDIATE NEURONS BEING USED TO PASS TO CLASSIFIER (or whatever h2t does exactly)...HARDCODE STUFF AS NEEDED